# Lab | Conectando Python con SQL

**Base de datos:** Sakila (alquiler de películas)  
**Objetivo:** Conectar Python con MySQL usando SQLAlchemy, y comparar la actividad
de los clientes entre dos meses distintos.

---
### Reto
Identificar qué clientes alquilaron películas tanto en mayo como en junio,
y cómo varió su número de alquileres entre ambos meses.

## Paso 1 — Importar librerías y conectar con la base de datos

Usamos `sqlalchemy` para crear el motor de conexión y `pymysql` como driver
para conectar Python con MySQL — la misma base `sakila` que usamos en MySQL Workbench.

In [ ]:
# Instalamos las librerías necesarias si no están disponibles
# !pip install sqlalchemy pymysql

In [ ]:
# Manipulación de datos
import pandas as pd

# Conexión a la base de datos
from sqlalchemy import create_engine

# Para introducir la contraseña sin que quede visible en el notebook
from getpass import getpass

print('Librerías importadas correctamente')

In [ ]:
# Pedimos la contraseña de MySQL de forma segura (no se guarda en el notebook)
password = getpass('Introduce la contraseña de MySQL: ')

# Datos de conexión
user = 'root'
host = 'localhost'
database = 'sakila'

# Creamos el string de conexión y el engine
connection_string = f'mysql+pymysql://{user}:{password}@{host}/{database}'
engine = create_engine(connection_string)

print('Conexión establecida correctamente')

In [ ]:
# Probamos la conexión con una consulta sencilla
test_query = 'SELECT COUNT(*) AS total_rentals FROM rental;'
pd.read_sql(test_query, engine)

## Paso 2 — Función `rentals_month`

Esta función recibe un mes y un año, y devuelve todos los alquileres
de la tabla `rental` que ocurrieron en ese mes y año como un DataFrame.

In [ ]:
def rentals_month(engine, month, year):
    """
    Recupera los alquileres realizados en un mes y año concretos.

    Parámetros:
    - engine: motor de conexión SQLAlchemy a la base de datos Sakila
    - month (int): mes a consultar (1-12)
    - year (int): año a consultar

    Devuelve:
    - DataFrame con los alquileres de ese mes y año
    """
    query = f"""
        SELECT *
        FROM rental
        WHERE MONTH(rental_date) = {month}
          AND YEAR(rental_date) = {year};
    """

    df = pd.read_sql(query, engine)
    return df

In [ ]:
# Probamos la función para mayo de 2005
rentals_may = rentals_month(engine, month=5, year=2005)

print(f'Alquileres en mayo 2005: {rentals_may.shape[0]}')
rentals_may.head()

In [ ]:
# Probamos la función para junio de 2005
rentals_june = rentals_month(engine, month=6, year=2005)

print(f'Alquileres en junio 2005: {rentals_june.shape[0]}')
rentals_june.head()

## Paso 3 — Función `rental_count_month`

Esta función recibe el DataFrame de un mes y devuelve el número de alquileres
por `customer_id`, nombrando la nueva columna como `rentals_MM_YYYY`.

In [ ]:
def rental_count_month(df, month, year):
    """
    Cuenta el número de alquileres por cliente para un mes y año concretos.

    Parámetros:
    - df: DataFrame devuelto por rentals_month()
    - month (int): mes correspondiente a los datos
    - year (int): año correspondiente a los datos

    Devuelve:
    - DataFrame con columnas: customer_id, rentals_MM_YYYY
    """
    # Construimos el nombre de la columna con formato MM_YYYY
    column_name = f'rentals_{month:02d}_{year}'

    # Agrupamos por customer_id y contamos el número de alquileres
    result = df.groupby('customer_id').size().reset_index(name=column_name)

    return result

In [ ]:
# Aplicamos la función a los datos de mayo y junio
rentals_may_count = rental_count_month(rentals_may, month=5, year=2005)
rentals_june_count = rental_count_month(rentals_june, month=6, year=2005)

print('Conteo de alquileres - Mayo 2005:')
print(rentals_may_count.head())

print('\nConteo de alquileres - Junio 2005:')
print(rentals_june_count.head())

## Paso 4 — Función `compare_rentals`

Esta función combina los dos DataFrames de conteo mensual y calcula
la diferencia de alquileres entre ambos meses para cada cliente.

In [ ]:
def compare_rentals(df1, df2):
    """
    Combina dos DataFrames de conteo de alquileres mensuales y calcula
    la diferencia entre ambos meses para cada cliente.

    Parámetros:
    - df1: DataFrame de conteo de alquileres del primer mes
    - df2: DataFrame de conteo de alquileres del segundo mes

    Devuelve:
    - DataFrame combinado con una columna 'difference' (df2 - df1)
    """
    # Combinamos por customer_id, manteniendo todos los clientes de ambos meses
    combined = df1.merge(df2, on='customer_id', how='outer')

    # Rellenamos con 0 los clientes que no alquilaron en alguno de los dos meses
    combined.fillna(0, inplace=True)

    # Identificamos las columnas de conteo (todas excepto customer_id)
    count_columns = [col for col in combined.columns if col != 'customer_id']

    # Calculamos la diferencia: segundo mes - primer mes
    combined['difference'] = combined[count_columns[1]] - combined[count_columns[0]]

    return combined

In [ ]:
# Comparamos la actividad de los clientes entre mayo y junio de 2005
comparison = compare_rentals(rentals_may_count, rentals_june_count)

print(f'Total de clientes: {comparison.shape[0]}')
comparison.head(10)

## Análisis de resultados

Con la columna `difference` podemos identificar:
- Clientes que **aumentaron** su actividad de mayo a junio (difference > 0)
- Clientes que **disminuyeron** su actividad (difference < 0)
- Clientes que mantuvieron el mismo nivel de actividad (difference == 0)

In [ ]:
# Clientes que más incrementaron su actividad
print('Top 5 clientes que MÁS incrementaron sus alquileres de mayo a junio:')
comparison.sort_values('difference', ascending=False).head()

In [ ]:
# Clientes que más redujeron su actividad
print('Top 5 clientes que MÁS redujeron sus alquileres de mayo a junio:')
comparison.sort_values('difference', ascending=True).head()

In [ ]:
# Resumen general de la comparación
print('=== RESUMEN ===')
print(f'Clientes activos en mayo:           {(comparison.iloc[:, 1] > 0).sum()}')
print(f'Clientes activos en junio:          {(comparison.iloc[:, 2] > 0).sum()}')
print(f'Clientes activos en ambos meses:    {((comparison.iloc[:, 1] > 0) & (comparison.iloc[:, 2] > 0)).sum()}')
print(f'\nClientes que aumentaron actividad: {(comparison["difference"] > 0).sum()}')
print(f'Clientes que redujeron actividad:   {(comparison["difference"] < 0).sum()}')
print(f'Clientes sin cambio:                {(comparison["difference"] == 0).sum()}')
print(f'\nDiferencia media: {comparison["difference"].mean():.2f}')

## Conclusiones

- Conectar Python con MySQL mediante `sqlalchemy` permite ejecutar consultas SQL
  directamente sobre la base de datos y trabajar con los resultados como DataFrames de pandas.
- Separar la lógica en funciones reutilizables (`rentals_month`, `rental_count_month`, `compare_rentals`)
  permite repetir el análisis para cualquier par de meses y años sin duplicar código.
- El uso de `merge(how='outer')` junto con `fillna(0)` es clave para no perder clientes
  que solo estuvieron activos en uno de los dos meses comparados.
- La columna `difference` permite identificar rápidamente patrones de cambio en el
  comportamiento de los clientes — información útil para campañas de retención o fidelización.